*전체 11단계 중 2단계(데이터 수집)의 노트북입니다 — 전력거래소 발전원별 발전량 API. 진행 기록은 리포의 `README.md`.*

# 2단계 수집 — 발전원별 발전량 API (하루 한 번 실행)

**무엇을 하나** — 공공데이터포털 「한국전력거래소_발전원별 발전량」 API로 하루치(연료 8종 × 24시간 = 192행)씩 받아 `data/raw/api/YYYYMMDD.csv`로 저장한다.
**왜 하루 한 번인가** — 개발계정은 하루 100건 제한이라, 한 번 실행에 95일치까지만 받는다. 이미 받은 날짜는 건너뛰므로 며칠에 걸쳐 그냥 다시 실행하면 이어서 받는다.
**지금 기간** — 2026-01-01 ~ 2026-07-22. 1월 1일은 이미 받아 두었으니 202일이 남아 세 번 실행하면 끝난다.
**정의 대조는 끝났다 — 이제 수집 셀만 돌리면 된다.**
**0행으로 온 날**은 빈 파일로 남는데, 다음 실행 때 자동으로 한 번 더 시도한다. 그래도 0행이면 API에 그날이 없는 것이다.

---

**확인이 끝난 것 (2026-09-13, 실측)**
1. **API의 LNG = 연간 파일의 LNG** — 2025-01-01 24시간 전부, 절대 차이 최대 0.001 MWh.
2. **API의 「유전」 = 파일의 「중유」** — 차이 0.0. 따라서 **「신재생·기타」 = 파일에서 LNG·무연탄·수력·양수·원자력·유연탄·중유를 뺀 나머지 전부**(시간별 최대 차이 87.4 MWh, 원인 확인 불가). 그 안에서 태양광이 27.3%다.

**데이터에 구멍 둘 (2026-09-15, 실측)**
3. **2024년 8월 1일부터 31일까지 한 달을 전부 못 쓴다.** 하루 120행(연료 5종)만 오고 원자력·무연탄·유전이 빠져 있다. 남은 연료의 값도 잘못돼 있다 — 8/1 LNG가 294~551 MWh인데 7/31은 9,273~29,332 MWh다. 09-15에 같은 요청을 다시 보내도 같고, 응답의 `totalCount`도 120이며 2페이지는 0행이다. **우리가 덜 받은 것이 아니라 API에 그렇게 들어 있다.** 7월 31일까지와 9월 1일부터는 정상이다. `03_정제`의 준비 셀(①②)이 지금은 192행이 아닌 날을 뺀다 — 뺄지 말지는 3단계 물음 1에서 내가 정한다.
4. **2026년은 7월 하순부터 아직 없다.** 1월 1일 ~ 7월 16일은 192행으로 오는데 7월 23·27·29·30·31일과 8월은 `totalCount 0`이다. 마지막 날은 7월 17~22일 사이다. 잘못된 것이 아니라 없는 것이므로 나중에 채워질 수 있다.
5. **2024년 연간 파일에는 LNG가 아예 없다.** 파일을 직접 읽어 13종(IGCC·수력·태양광 등 신재생 계열)만 있는 것을 확인했다. 그래서 2024년 LNG·원자력은 이 API뿐이다. 8월을 메울 다른 출처도 09-15에 전력거래소 데이터셋 열 개를 열어 봤으나 없었다 — 「연료원별 전력거래량」(연료원별 일별 시간대별)이 모양은 맞는데 2023년 12월까지뿐이고, 「발전원별 발전량(계통기준)」과 「발전원별 발전량 현황조회」는 5분 단위 현재값이라 과거 이력이 없다.

**기본값** — 학습 2023년 온해 + 2024년 334일 / 채점 2025년 온해 / 적용 2026년 1~6월. 셋 다 3·6단계에서 내가 다시 정한다.
**다음 순서** — 2026년 수집이 끝나면 맨 아래 「날짜별 건수 확인」 셀로 7·8월이 올라왔는지 보고, 올라왔으면 `END`를 늘려 더 받는다.

In [1]:
import os                                    # 폴더·파일 경로를 다루는 기본 모듈
import time                                  # 호출 사이에 잠깐 쉬려고 쓴다
import requests                              # 인터넷으로 API를 부르는 라이브러리
import pandas as pd                          # 표(데이터프레임)를 다루는 라이브러리
from dotenv import load_dotenv, find_dotenv  # .env 파일에서 키를 읽는 도구

load_dotenv(find_dotenv())                   # 프로젝트 안의 .env 파일을 찾아 읽는다 (키는 화면에 찍지 않는다)
KEY = os.environ["DATA_GO_KR_KEY"]           # 환경변수에서 인증키(Decoding 64자)를 꺼낸다

START = "2026-01-01"                         # 받을 첫 날짜 (2024년은 2026-09-15에 다 받았다)
END   = "2026-09-30"  # 받을 마지막 날짜 — 10-10에 9월 말까지 늘렸다. 9단계에서 비교 구간을 9월까지 늘릴지 정하려고 받아 둔다
LIMIT = 95                                   # 한 번 실행에 부를 최대 횟수 (하루 100건 제한, 여유 5건)

URL  = "https://apis.data.go.kr/B552115/PvAmountByPwrGen/getPvAmountByPwrGen"  # 발전원별 발전량 API 주소
ROOT = os.path.dirname(find_dotenv())        # .env가 있는 폴더 = 프로젝트 루트
OUT  = os.path.join(ROOT, "data", "raw", "api")  # 하루치 파일을 모을 폴더
os.makedirs(OUT, exist_ok=True)              # 폴더가 없으면 만든다

days = pd.date_range(START, END, freq="D")   # START~END 사이의 모든 날짜
done = {f[:8] for f in os.listdir(OUT)                            # 이미 받은 날짜 (파일명 앞 8자리 YYYYMMDD)
        if f.endswith(".csv") and os.path.getsize(os.path.join(OUT, f)) > 100}   # 단, 100바이트 미만(0행) 파일은 안 받은 것으로 쳐서 다시 시도한다
todo = [d for d in days if d.strftime("%Y%m%d") not in done]    # 아직 안 받은 날짜만 남긴다
print(f"전체 {len(days)}일 중 받은 것 {len(days) - len(todo)}일, 남은 것 {len(todo)}일")

def fetch_day(ymd):                          # 하루치(YYYYMMDD)를 받아 표로 돌려주는 함수
    params = {"serviceKey": KEY,             # 인증키
              "pageNo": 1,                   # 첫 페이지
              "numOfRows": 300,              # 한 번에 받을 최대 행 수 (하루는 192행)
              "dataType": "json",            # JSON으로 달라고 요청
              "tradeYmd": ymd}               # 받을 날짜
    r = requests.get(URL, params=params, timeout=30)   # API를 부른다 (30초 넘으면 포기)
    r.raise_for_status()                     # 응답 코드가 200이 아니면 여기서 오류를 낸다
    body = r.json()["response"]["body"]      # 응답에서 본문(body)만 꺼낸다
    items = body.get("items") or {}          # 데이터가 없는 날은 빈 딕셔너리
    rows = items.get("item") or []           # 행 목록 (없으면 빈 리스트)
    return pd.DataFrame(rows)                # 행 목록을 표로 바꿔 돌려준다

calls = 0                                    # 이번 실행에서 부른 횟수
for d in todo[:LIMIT]:                       # 남은 날짜 중 앞에서 LIMIT개만
    ymd = d.strftime("%Y%m%d")               # 날짜를 20240101 꼴 문자열로
    try:
        df = fetch_day(ymd)                  # 하루치 받기
    except requests.HTTPError as e:          # 서버가 오류 코드로 답한 경우 (429 = 오늘 한도 100건이 찬 것)
        print(f"{ymd} 실패: 상태 코드 {e.response.status_code} — 여기서 멈춘다. 429면 내일 다시 실행하면 이어서 받는다")  # 주소는 찍지 않는다 (주소 안에 키가 있다)
        break                                # 반복을 끝낸다
    except Exception as e:                   # 네트워크 끊김 등 그 밖의 실패
        print(f"{ymd} 실패: {type(e).__name__} — 여기서 멈춘다. 내일 다시 실행하면 이어서 받는다")  # 오류 종류 이름만 찍는다
        break                                # 반복을 끝낸다
    calls += 1                               # 호출 횟수 +1
    df.to_csv(os.path.join(OUT, f"{ymd}.csv"), index=False, encoding="utf-8-sig")  # 하루치 저장 (0행이어도 저장해 다시 안 부른다)
    if len(df) != 192:                       # 정상은 연료 8종 × 24시간 = 192행
        print(f"{ymd}: {len(df)}행 (192행이 아님 — 나중에 확인)")
    if calls % 10 == 0:                      # 열 번마다
        print(f"  {calls}번째 호출 — {ymd}까지 저장됨")   # 진행 상황을 한 줄 찍는다 (멈춘 게 아니라는 표시)
    time.sleep(0.3)                          # 서버에 부담 안 주게 0.3초 쉰다

print(f"이번 실행 호출 {calls}건 · 누적 파일 {len(os.listdir(OUT))}개 · 남은 것 {len(todo) - calls}일")


전체 273일 중 받은 것 226일, 남은 것 47일
20260517: 0행 (192행이 아님 — 나중에 확인)
20260823: 120행 (192행이 아님 — 나중에 확인)
20260824: 0행 (192행이 아님 — 나중에 확인)
  10번째 호출 — 20260824까지 저장됨
20260825: 0행 (192행이 아님 — 나중에 확인)
20260826: 0행 (192행이 아님 — 나중에 확인)
20260827: 0행 (192행이 아님 — 나중에 확인)
20260828: 0행 (192행이 아님 — 나중에 확인)
20260829: 0행 (192행이 아님 — 나중에 확인)
20260830: 0행 (192행이 아님 — 나중에 확인)
20260831: 0행 (192행이 아님 — 나중에 확인)
20260901: 0행 (192행이 아님 — 나중에 확인)
20260902: 0행 (192행이 아님 — 나중에 확인)
20260903: 0행 (192행이 아님 — 나중에 확인)
  20번째 호출 — 20260903까지 저장됨
20260904: 0행 (192행이 아님 — 나중에 확인)
20260905: 0행 (192행이 아님 — 나중에 확인)
20260906: 0행 (192행이 아님 — 나중에 확인)
20260907: 0행 (192행이 아님 — 나중에 확인)
20260908: 0행 (192행이 아님 — 나중에 확인)
20260909: 0행 (192행이 아님 — 나중에 확인)
20260910: 0행 (192행이 아님 — 나중에 확인)
20260911: 0행 (192행이 아님 — 나중에 확인)
20260912: 0행 (192행이 아님 — 나중에 확인)
20260913: 0행 (192행이 아님 — 나중에 확인)
  30번째 호출 — 20260913까지 저장됨
20260914: 0행 (192행이 아님 — 나중에 확인)
20260915: 0행 (192행이 아님 — 나중에 확인)
20260916: 0행 (192행이 아님 — 나중에 확인)
20260917: 0행 (192행이 아님 — 나중에 

## 정의 대조 — API와 연간 파일이 같은 값인가 (호출 1건, 끝났다)

2024·2026년은 API로, 2023·2025년은 연간 파일로 받는다. 2025-01-01 하루로 둘을 맞춰 봤고 **결과는 맨 위 머리말에 적어 두었다.** 요약하면 LNG는 차이 0.001 MWh, 「유전」은 파일의 「중유」, 「신재생·기타」는 나머지 전부(태양광이 그 27.3%)다.

**다시 돌리지 않는다.** 돌리면 호출 1건을 또 쓴다. 아래 코드는 어떻게 확인했는지 남겨 두는 기록이다.

In [1]:
import os, unicodedata, requests, pandas as pd                  # 파일 목록·한글 정규화·API 호출·표 처리
from dotenv import load_dotenv, find_dotenv                      # .env에서 키를 읽는 도구
load_dotenv(find_dotenv())                                       # .env를 찾아 환경변수로 올린다
KEY = os.environ["DATA_GO_KR_KEY"]                              # 키 (화면에 찍지 않는다)
ROOT = os.path.dirname(find_dotenv())                            # 프로젝트 뿌리 = .env가 있는 폴더
URL = "https://apis.data.go.kr/B552115/PvAmountByPwrGen/getPvAmountByPwrGen"   # 발전원별 발전량 API 주소
params = {"serviceKey": KEY, "pageNo": 1, "numOfRows": 300, "dataType": "json", "tradeYmd": "20250101"}  # 2025-01-01 하루
r = requests.get(URL, params=params, timeout=30)                 # 호출한다
if r.status_code != 200:                                         # 200이 아니면 실패
    raise SystemExit(f"실패: 상태 코드 {r.status_code} (429면 오늘 한도가 찬 것 — 내일 수집 셀보다 먼저 실행)")  # 주소는 찍지 않는다 (주소 안에 키가 있다)
rows = (r.json()["response"]["body"].get("items") or {}).get("item") or []   # 응답에서 행 목록만 꺼낸다
api = pd.DataFrame(rows)                                         # 표로 만든다 (192행 = 연료 8종 × 24시간)
api_lng = api[api.fuelTpCd == "LNG"].astype({"tradeNo": int, "amgo": float}).sort_values("tradeNo")  # LNG 24행만, 시간 순
kpx_dir = os.path.join(ROOT, "data", "raw", "kpx")               # 연간 파일 폴더
f2025 = [f for f in os.listdir(kpx_dir) if "2025" in unicodedata.normalize("NFC", f)][0]  # 2025 파일 이름 (맥 한글 NFD 대비)
kpx = pd.read_csv(os.path.join(kpx_dir, f2025), encoding="utf-8-sig")        # 2025 연간 파일
kpx.columns = ["date", "hour", "fuel", "cap_mw", "mwh"]          # 열 이름을 짧게
kpx_lng = kpx[(kpx.fuel == "LNG") & (kpx.date == "2025-01-01")].sort_values("hour")  # 같은 날 LNG 24행
assert len(api_lng) == 24 and len(kpx_lng) == 24, (len(api_lng), len(kpx_lng))       # 둘 다 24행이어야 비교가 된다
cmp = pd.DataFrame({"hour": range(1, 25), "api_MWh": api_lng.amgo.values, "file_MWh": kpx_lng.mwh.values})  # 나란히 놓는다
cmp["diff"] = cmp.api_MWh - cmp.file_MWh                         # 시간별 차이
print(cmp.round(2).to_string(index=False))                       # 24줄 표
print("절대 차이 최대:", round(cmp["diff"].abs().max(), 3), "MWh")  # 0에 가까우면 같은 정의

# ── 이어서: API의 「신재생·기타」에 무엇이 들었나 (추가 호출 없이 위 응답을 그대로 쓴다) ──
api = api.astype({"tradeNo": int, "amgo": float})                  # 시간·발전량을 숫자로 바꾼다
api_h = api.pivot_table(index="tradeNo", columns="fuelTpCd", values="amgo")   # 시간(1~24) × 연료 표
kpx_day = kpx[kpx.date == "2025-01-01"]                            # 같은 날, 파일 쪽은 연료 25~26종
file_h = kpx_day.pivot_table(index="hour", columns="fuel", values="mwh")      # 파일도 같은 모양으로
print("\n[API] 그날 연료별 합계(MWh)"); print(api_h.sum().round(0).to_string())
print("\n[파일] 그날 연료별 합계(MWh)"); print(file_h.sum().round(0).sort_values(ascending=False).to_string())

same = ["LNG", "무연탄", "수력", "양수", "원자력", "유연탄"]        # 두 자료에 이름이 그대로 있는 여섯
print("\n이름이 같은 여섯 — 값도 같은지")
for f in same:                                                     # 하나씩 시간별로 뺀다
    print(f"  {f}: 절대 차이 최대 {abs(api_h[f].values - file_h[f].values).max():.2f} MWh")

rest = [c for c in file_h.columns if c not in same]                # 파일에서 남은 연료들
oil_col = [c for c in api_h.columns if "유전" in c][0]              # API의 유류 열 이름
new_col = [c for c in api_h.columns if "신재생" in c][0]            # API의 신재생 열 이름
print("\nAPI의 「유전」이 파일의 무엇인지 후보를 대 본다 — 차이가 0에 가까운 줄이 정답")
for g in [["중유", "경유"], ["중유"], ["경유"]]:                     # 후보 셋
    g = [x for x in g if x in file_h.columns]                      # 그날 파일에 없는 연료는 뺀다
    oil = file_h[g].sum(axis=1)                                    # 유류로 볼 연료들의 합
    new = file_h[[c for c in rest if c not in g]].sum(axis=1)      # 나머지 전부 = 신재생·기타로 볼 값
    d1 = abs(api_h[oil_col].values - oil.values).max()             # 유전 쪽 시간별 최대 차이
    d2 = abs(api_h[new_col].values - new.values).max()             # 신재생 쪽 시간별 최대 차이
    print(f"  유전={g} → 유전 차이 {d1:.1f} · 신재생·기타 차이 {d2:.1f} MWh")

print("\n태양광이 그 안에서 차지한 몫:", round(file_h["태양광"].sum() / file_h[rest].sum(axis=1).sum() * 100, 1), "%")


 hour  api_MWh  file_MWh  diff
    1 22446.55  22446.55   0.0
    2 20142.84  20142.84  -0.0
    3 19044.71  19044.71  -0.0
    4 19511.00  19511.00  -0.0
    5 19143.76  19143.76   0.0
    6 18571.94  18571.94  -0.0
    7 18810.70  18810.70   0.0
    8 19314.90  19314.90  -0.0
    9 18693.58  18693.59  -0.0
   10 16120.90  16120.90  -0.0
   11 13769.03  13769.03  -0.0
   12 11671.45  11671.45   0.0
   13 11634.73  11634.73  -0.0
   14 11436.14  11436.14  -0.0
   15 12708.80  12708.80   0.0
   16 15818.83  15818.83   0.0
   17 19839.30  19839.30  -0.0
   18 23484.91  23484.91   0.0
   19 23320.82  23320.82  -0.0
   20 22460.16  22460.16  -0.0
   21 21599.64  21599.64   0.0
   22 20795.76  20795.76  -0.0
   23 20180.84  20180.84  -0.0
   24 19582.25  19582.25  -0.0
절대 차이 최대: 0.001 MWh

[API] 그날 연료별 합계(MWh)
fuelTpCd
LNG       440104.0
무연탄         3292.0
수력          5811.0
신재생·기타    108443.0
양수         11567.0
원자력       527534.0
유연탄       244548.0
유전           835.0

[파일] 그날 연료별 합계(MWh)
f

In [ ]:
# 날짜별로 API에 몇 건이 있는지만 본다 — 7·8월분이 올라왔는지 볼 때 쓴다 (날짜 하나당 호출 1건)
import os                                     # 폴더·파일 경로를 다루는 기본 모듈
import requests                               # 인터넷으로 API를 부르는 라이브러리
from dotenv import load_dotenv, find_dotenv   # .env 파일에서 키를 읽는 도구

load_dotenv(find_dotenv())                    # 프로젝트 안의 .env 파일을 찾아 읽는다
KEY = os.environ["DATA_GO_KR_KEY"]            # 환경변수에서 인증키를 꺼낸다
URL = "https://apis.data.go.kr/B552115/PvAmountByPwrGen/getPvAmountByPwrGen"  # 발전원별 발전량 API 주소

볼날 = ["20260731", "20260831", "20260930"]    # 7 · 8 · 9월 끝날이 API에 올라왔는지 본다 — 수집 셀보다 먼저 돌린다 (한 날짜당 호출 1건)

for ymd in 볼날:                                # 날짜를 하나씩
    params = {"serviceKey": KEY,               # 인증키
              "pageNo": 1,                     # 첫 페이지
              "numOfRows": 300,                # 한 번에 받을 최대 행 수
              "dataType": "json",              # JSON으로 달라고 요청
              "tradeYmd": ymd}                 # 볼 날짜
    try:
        r = requests.get(URL, params=params, timeout=30)   # API를 부른다 (30초 넘으면 포기)
    except Exception as e:                     # 네트워크가 끊긴 경우
        print(ymd, "실패:", type(e).__name__); continue    # 오류 종류 이름만 찍는다
    if r.status_code != 200:                   # 서버가 오류 코드로 답한 경우 (429 = 하루 한도 초과)
        print(ymd, "실패: 상태 코드", r.status_code); continue   # 주소는 찍지 않는다 (주소 안에 키가 있다)
    body = r.json()["response"]["body"]        # 응답에서 본문만 꺼낸다
    n = int(body.get("totalCount") or 0)       # API가 적어 보낸 건수 — 글자로 오므로 int로 바꿔야 한다
    말 = "정상" if n == 192 else ("아직 없음" if n == 0 else "이상 — 연료가 빠졌다")  # 세 가지로 가른다
    print(f"{ymd} → {n}건   {말}")              # 한 줄로 찍는다


In [2]:
# 전체 11단계 중 → 2단계 데이터 수집과 구성 보기 · 2026년 5월 2~17일을 다시 요청해 지금 파일과 같은지 본다 (호출 16건)
import os                                        # 파일 경로를 다루는 기본 모듈
import time                                      # 호출 사이에 잠깐 쉬려고 쓴다
import requests                                  # 인터넷으로 API를 부르는 라이브러리
import pandas as pd                              # 표를 다루는 라이브러리
from dotenv import load_dotenv, find_dotenv      # .env에서 키를 읽는 도구

load_dotenv(find_dotenv())                       # .env를 찾아 읽는다 (키는 화면에 찍지 않는다)
KEY  = os.environ["DATA_GO_KR_KEY"]              # 인증키를 꺼낸다
URL  = "https://apis.data.go.kr/B552115/PvAmountByPwrGen/getPvAmountByPwrGen"   # 발전원별 발전량 API 주소
OUT  = os.path.join(os.path.dirname(find_dotenv()), "data", "raw", "api")       # 지금 갖고 있는 하루치 파일 폴더

def 요약(df):                                     # 표 하나에서 행 수·연료 종수·LNG 하루합을 뽑는다
    if df is None or len(df) == 0:
        return 0, 0, 0.0
    lng = df[df["fuelTpCd"] == "LNG"]["amgo"].astype(float).sum()
    return len(df), df["fuelTpCd"].nunique(), round(float(lng))

줄 = []                                           # 결과를 한 줄씩 모은다
for 날 in pd.date_range("2026-05-02", "2026-05-17").strftime("%Y%m%d"):
    p = {"serviceKey": KEY, "pageNo": 1, "numOfRows": 300, "dataType": "json", "tradeYmd": 날}
    r = requests.get(URL, params=p, timeout=30)   # 호출한다
    if r.status_code != 200:                      # 200이 아니면 멈춘다 (주소는 찍지 않는다 — 주소 안에 키가 있다)
        print(f"{날} 실패 — 상태 코드 {r.status_code} (429면 오늘 한도가 찼다)")
        break
    몸통 = r.json()["response"]["body"]            # 응답의 본문
    새 = pd.DataFrame((몸통.get("items") or {}).get("item") or [])                # 이번에 받은 표
    새행, 새연료, 새LNG = 요약(새)
    옛길 = os.path.join(OUT, f"{날}.csv")          # 지금 갖고 있는 파일
    옛 = pd.read_csv(옛길) if os.path.exists(옛길) and os.path.getsize(옛길) >= 100 else None
    옛행, 옛연료, 옛LNG = 요약(옛)
    줄.append([날, int(몸통.get("totalCount") or 0), 옛행, 새행, 옛LNG, 새LNG,
               "같다" if (옛행 == 새행 and 옛LNG == 새LNG) else "다르다"])
    time.sleep(0.3)                               # 연속 호출 사이에 잠깐 쉰다

표 = pd.DataFrame(줄, columns=["날짜", "API가_말하는_행수", "지금_행수", "다시받은_행수",
                               "지금_LNG합", "다시받은_LNG합", "같은가"])
print("2026년 5월 2~17일을 다시 요청한 결과 (LNG 합은 MWh)\n")
print(표.to_string(index=False))
print("\n다 「같다」이면 API에 그렇게 들어 있는 것이고, 하나라도 「다르다」이면 그날은 다시 받을 수 있다.")

20260507 실패 — 상태 코드 429 (429면 오늘 한도가 찼다)
2026년 5월 2~17일을 다시 요청한 결과 (LNG 합은 MWh)

      날짜  API가_말하는_행수  지금_행수  다시받은_행수  지금_LNG합  다시받은_LNG합 같은가
20260502          120    120      120     1047       1047  같다
20260503           48     48       48        0          0  같다
20260504           48     48       48        0          0  같다
20260505           48     48       48        0          0  같다
20260506           48     48       48        0          0  같다

다 「같다」이면 API에 그렇게 들어 있는 것이고, 하나라도 「다르다」이면 그날은 다시 받을 수 있다.
